# 06. 웹 애플리케이션 배포 (FastAPI + Streamlit + ngrok)

구축한 시스템을 외부 사용자가 접속 가능한 웹 애플리케이션으로 노출합니다.

**소요 시간**: 시작 후 즉시 사용 가능 (Colab 세션 동안)

## 6.1 환경 설정

In [ ]:
import os, sys, time, subprocess
from google.colab import userdata
sys.path.insert(0, '/content/repo')

# 모든 API 키 + ngrok 토큰
required_keys = [
    'OPENAI_API_KEY',
    'CLOVA_API_KEY',
    'NAVER_CLIENT_ID',
    'NAVER_CLIENT_SECRET',
    'NGROK_AUTHTOKEN',
]

for key in required_keys:
    try:
        os.environ[key] = userdata.get(key)
        print(f"  ✅ {key}")
    except:
        print(f"  ❌ {key} (Colab Secrets에 등록 필요)")
        raise

## 6.2 ngrok 설정

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])

# 기존 터널 정리
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

print(" ngrok 토큰 등록 + 기존 터널 정리")

## 6.3 FastAPI 백엔드 시작 (포트 8000)

In [ ]:
# 기존 프로세스 정리
!pkill -9 -f api_server 2>/dev/null
time.sleep(2)

# 백엔드 시작
api_proc = subprocess.Popen(
    ['python', '-m', 'src.server.api_server', '--no-ngrok'],
    cwd='/content/repo',
    stdout=open('/content/api.log', 'w'),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
print(f"FastAPI PID: {api_proc.pid}")

# Health 체크
time.sleep(8)
import requests
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f" FastAPI 정상: {r.json()}")
except Exception as e:
    print(f" FastAPI 실패: {e}")
    !tail -30 /content/api.log

## 6.4 Streamlit 프론트엔드 시작 (포트 8501)

In [ ]:
# 기존 프로세스 정리
!pkill -9 -f streamlit 2>/dev/null
time.sleep(2)

# Streamlit 시작 (ngrok 호환 옵션)
streamlit_env = {**os.environ, 'API_URL': 'http://localhost:8000'}
streamlit_proc = subprocess.Popen(
    [
        'streamlit', 'run', 'src/server/streamlit_app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.address=0.0.0.0',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--browser.gatherUsageStats=false',
    ],
    cwd='/content/repo',
    stdout=open('/content/streamlit.log', 'w'),
    stderr=subprocess.STDOUT,
    env=streamlit_env,
    start_new_session=True,
)
print(f"Streamlit PID: {streamlit_proc.pid}")

# 부팅 대기
time.sleep(15)

try:
    r = requests.get('http://localhost:8501', timeout=5)
    print(f" Streamlit 정상: status {r.status_code}")
except Exception as e:
    print(f"실패 {e}")
    !tail -15 /content/streamlit.log

## 6.5 ngrok 터널 생성 (외부 노출)

In [ ]:
# Streamlit(8501)을 외부에 노출
public_url = ngrok.connect(8501, 'http')

print('=' * 60)
print('  웹 애플리케이션 외부 접속 URL')
print('=' * 60)
print(f"\n  {public_url.public_url}\n")
print('=' * 60)

## 6.6 시스템 상태 점검

In [ ]:
# 종합 점검
print("=" * 60)
print("  시스템 상태")
print("=" * 60)

# 1. FastAPI
try:
    r = requests.get('http://localhost:8000/health', timeout=5).json()
    print(f"[✅] FastAPI:    {r['status']}")
    print(f"     Facts DB:    {r.get('facts_db', '-')}")
    print(f"     Pipeline:    {r.get('pipeline_loaded', '-')} (lazy load)")
except Exception as e:
    print(f"[❌] FastAPI: {e}")

# 2. Streamlit
try:
    r = requests.get('http://localhost:8501', timeout=5)
    print(f"[✅] Streamlit:  status {r.status_code}")
except Exception as e:
    print(f"[❌] Streamlit: {e}")

# 3. ngrok
tunnels = ngrok.get_tunnels()
if tunnels:
    for t in tunnels:
        print(f"[✅] ngrok:      {t.public_url}")
else:
    print(f"[❌] ngrok: 활성 터널 없음")

## 6.7 시연 워밍업 (선택)

첫 narrative 호출은 GPU 모델 로딩으로 60-90초 소요. 시연 전에 미리 한 번 호출하면 이후 응답이 빠릅니다.

In [ ]:
# narrative 모델 사전 로딩
print("Pipeline warmup 중... (약 60-90초)")
t0 = time.time()
r = requests.post(
    'http://localhost:8000/ask',
    json={'question': '삼성전자 사업 부문은?'},
    timeout=180,
)
elapsed = time.time() - t0
print(f"  Warmup: {elapsed:.1f}초")
print(f" 이후 narrative 호출은 5-15초로 빠릅니다")

## 6.8 종료 (시연 완료 후)

Colab 무료 사용량 절약을 위해 사용 완료 시 정리:

In [ ]:
# ngrok 터널 종료
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
print(" ngrok 종료")

# 프로세스 종료
!pkill -9 -f api_server 2>/dev/null
!pkill -9 -f streamlit 2>/dev/null
print(" 모든 서버 종료")

##  완료

전체 시스템이 작동 중입니다.

**다음 단계**:
- 보고서 작성 → `docs/연구논문작품_최종보고서.hwp` 참조
- 발표 자료 → `docs/교내발표용_PPT_2019312922.pptx` 참조